# Implement Ring Attention for Long Contexts — Solution

**Difficulty**: 🟣 Expert

**Companies**: Anthropic, Google, Meta, xAI

---

### Problem Statement

For extremely long sequences (100K+ tokens), even FlashAttention runs out of memory: it removes the N×N score matrix, but each device still holds the activations (Q, K, V, outputs) of the **whole sequence** in every layer. **Ring Attention** (Liu, Zaharia & Abbeel, 2023) shards the *sequence* across devices arranged in a **ring** (sequence/context parallelism).

The key idea:
- The sequence is split across `num_devices` devices. Device `i` holds Q chunk `i`, K chunk `i`, and V chunk `i`.
- Each device computes attention of its local Q against the **current** K,V chunk.
- Then K,V chunks are **passed to the next device** in the ring (device 0 -> 1 -> 2 -> ... -> 0).
- After `num_devices` ring steps, every Q chunk has attended to ALL K,V chunks.
- **Online softmax** is used to correctly combine partial attention results across ring steps.

This means each device only needs memory for:
- Its local Q chunk (permanent)
- The K,V chunk it is computing on, plus the one arriving from its neighbor (double buffering)
- Running softmax statistics (max, sum, output accumulator)

Peak memory per device: `O(N / num_devices)` instead of `O(N)`.

On real hardware each device sends its current K,V chunk to the next device *while* it computes with it. When a chunk is long enough that the blockwise attention takes longer than the transfer, the communication is fully hidden. With causal masking, plain contiguous chunks are load-imbalanced: device 0 skips almost every step while the last device computes at every step. Striped (token t on device t mod n) or zig-zag (each device holds one chunk from each end of the sequence) layouts fix that; Llama 3's context parallelism uses the zig-zag split, though it all-gathers K/V instead of passing chunks around a ring.

---

### Requirements

1. **Sequence Splitting** — Split Q, K, V across `num_devices` along the sequence dimension.
2. **Ring Step** — Implement `ring_step()` that computes attention of local Q against a received K,V chunk, updating online softmax statistics.
3. **Ring Communication** — Simulate the ring by rotating K,V chunks: device `i` sends to device `(i+1) % num_devices`.
4. **Causal Masking** — A K,V chunk that lies entirely in the future of the Q chunk (index `j > i`) is skipped. The diagonal chunk (`j == i`) gets a triangular mask, and past chunks (`j < i`) need no mask.
5. **Correctness** — Output must match standard full attention.

---

### Constraints

- ✅ Simulate multi-device with lists of tensors (no actual distributed communication)
- ✅ Must work on CPU
- ✅ Must handle causal and non-causal attention
- ✅ Sequence lengths that are not a multiple of `num_devices` work too (chunk sizes may differ by one)
- ❌ No device should ever hold more than two K,V chunks (about `2/num_devices` of the full K,V) at a time: the one being computed on and the one being received

---

<details>
  <summary>💡 Hint</summary>

  **Ring step with online softmax:**
  ```
  def ring_step(local_q, received_k, received_v, running_max, running_sum, running_output, scale):
      # Compute attention scores for this K,V chunk
      S = local_q @ received_k^T * scale     # (chunk_len, chunk_len)
      
      # Online softmax update (same as FlashAttention)
      block_max = S.max(dim=-1, keepdim=True)
      new_max = max(running_max, block_max)
      correction = exp(running_max - new_max)
      P = exp(S - new_max)
      
      running_sum = running_sum * correction + P.sum(dim=-1, keepdim=True)
      running_output = running_output * correction + P @ received_v
      running_max = new_max
      
      return running_max, running_sum, running_output
  ```
  
  **Ring rotation:** After each step, K,V chunks shift: `new_kv[i] = old_kv[(i-1) % n]`
  
  **Causal masking:** Device `i` with Q chunk `i` should skip K,V from chunk `j` if `j > i`
  (those are future tokens that Q should not attend to).

</details>

---

In [ ]:
import torch
import torch.nn.functional as F
import math
from typing import List, Tuple, Optional

In [ ]:
# Test data
torch.manual_seed(42)

# Dimensions
seq_len = 64        # total sequence length
head_dim = 32       # dimension per head
num_devices = 4     # number of simulated devices
chunk_len = seq_len // num_devices  # tokens per device

# Create Q, K, V: (seq_len, head_dim) — single head for clarity
Q = torch.randn(seq_len, head_dim)
K = torch.randn(seq_len, head_dim)
V = torch.randn(seq_len, head_dim)

print(f"Sequence length: {seq_len}")
print(f"Number of devices: {num_devices}")
print(f"Chunk length per device: {chunk_len}")
print(f"Head dimension: {head_dim}")
print(f"\nQ shape: {Q.shape}")
print(f"K shape: {K.shape}")
print(f"V shape: {V.shape}")

In [ ]:
def standard_attention(Q, K, V, causal=False):
    """
    Standard full attention: O(N^2) memory.
    
    Args:
        Q, K, V: (seq_len, head_dim)
        causal: whether to apply causal masking
    Returns:
        output: (seq_len, head_dim)
    """
    d = Q.shape[-1]
    scores = Q @ K.T / math.sqrt(d)  # (N, N)
    
    if causal:
        N = Q.shape[0]
        mask = torch.triu(torch.ones(N, N, dtype=torch.bool), diagonal=1)
        scores = scores.masked_fill(mask, float('-inf'))
    
    attn_weights = torch.softmax(scores, dim=-1)  # (N, N)
    output = attn_weights @ V  # (N, d)
    return output

In [ ]:
def ring_step(
    local_q: torch.Tensor,
    received_k: torch.Tensor,
    received_v: torch.Tensor,
    running_max: torch.Tensor,
    running_sum: torch.Tensor,
    running_output: torch.Tensor,
    scale: float,
    causal_mask: Optional[torch.Tensor] = None,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    One step of ring attention: compute attention of local_q against received_k, received_v
    and update the online softmax accumulators.
    
    Args:
        local_q: (q_len, head_dim) -- this device's Q chunk
        received_k: (kv_len, head_dim) -- K chunk received from ring
        received_v: (kv_len, head_dim) -- V chunk received from ring
        running_max: (chunk_len, 1) -- current running max for online softmax
        running_sum: (chunk_len, 1) -- current running sum for online softmax
        running_output: (chunk_len, head_dim) -- accumulated output
        scale: 1/sqrt(d)
        causal_mask: optional (chunk_len, chunk_len) bool mask, True = masked
    Returns:
        updated (running_max, running_sum, running_output)
    """
    # Compute attention scores for this K,V chunk
    S = local_q @ received_k.T * scale  # (q_len, kv_len)
    
    # Apply causal mask if provided (set masked positions to -inf)
    if causal_mask is not None:
        S = S.masked_fill(causal_mask, float('-inf'))
    
    # Online softmax update
    # 1. Find block max
    block_max = S.max(dim=-1, keepdim=True).values  # (chunk_len, 1)
    
    # 2. New running max
    new_max = torch.maximum(running_max, block_max)
    
    # 3. Correction factor for previously accumulated values
    correction = torch.exp(running_max - new_max)  # (chunk_len, 1)
    
    # 4. Local (unnormalized) attention weights
    P = torch.exp(S - new_max)  # (chunk_len, chunk_len)
    
    # 5. Update running sum
    running_sum = running_sum * correction + P.sum(dim=-1, keepdim=True)
    
    # 6. Update running output
    running_output = running_output * correction + P @ received_v
    
    # 7. Update running max
    running_max = new_max
    
    return running_max, running_sum, running_output

In [ ]:
class RingAttention:
    """
    Ring Attention: distributes sequence across devices in a ring topology.
    Each device computes local attention, then passes K,V to the next device.
    """

    def __init__(self, num_devices: int):
        self.num_devices = num_devices

    def _split_sequence(self, tensor: torch.Tensor) -> List[torch.Tensor]:
        """
        Split a (seq_len, head_dim) tensor into num_devices contiguous chunks along
        dim=0. tensor_split always returns exactly num_devices chunks (sizes differ
        by at most one), unlike torch.chunk, which can return fewer.
        """
        return list(torch.tensor_split(tensor, self.num_devices, dim=0))

    def _rotate_ring(self, chunks: List[torch.Tensor]) -> List[torch.Tensor]:
        """
        Rotate chunks in the ring: device i sends to device (i+1) % n.
        After rotation, device i holds what was on device (i-1) % n.

        Example with 4 devices:
        Before: [chunk_0, chunk_1, chunk_2, chunk_3]
        After:  [chunk_3, chunk_0, chunk_1, chunk_2]
        (device 0 now has chunk_3, device 1 has chunk_0, etc.)
        """
        # Last element wraps to the front
        return [chunks[-1]] + chunks[:-1]

    def forward(self, Q, K, V, causal=False):
        """
        Compute ring attention.

        Args:
            Q, K, V: (seq_len, head_dim)
            causal: whether to apply causal masking
        Returns:
            output: (seq_len, head_dim)
        """
        N, d = Q.shape
        assert N >= self.num_devices, "need at least one token per device"
        scale = 1.0 / math.sqrt(d)

        # Split Q, K, V across devices
        q_chunks = self._split_sequence(Q)
        k_chunks = self._split_sequence(K)
        v_chunks = self._split_sequence(V)

        # Online softmax accumulators for each device, sized by its own Q chunk
        acc = dict(dtype=Q.dtype, device=Q.device)
        running_max = [torch.full((q.shape[0], 1), float('-inf'), **acc) for q in q_chunks]
        running_sum = [torch.zeros((q.shape[0], 1), **acc) for q in q_chunks]
        running_output = [torch.zeros((q.shape[0], d), **acc) for q in q_chunks]

        # Track which original chunk index each device currently holds for K,V
        # Initially, device i holds its own K,V chunk (chunk index i)
        kv_source = list(range(self.num_devices))

        # Ring loop: num_devices steps
        # After all steps, each Q chunk will have attended to ALL K,V chunks
        for step in range(self.num_devices):
            # Each device processes its Q against the current K,V it holds
            # (on real hardware these run in parallel, overlapped with the transfer)
            for device_id in range(self.num_devices):
                q_chunk_idx = device_id              # Q chunk index (permanent)
                kv_chunk_idx = kv_source[device_id]  # K,V chunk currently held

                # Causal masking, per chunk pair:
                # - KV chunk entirely in the future (kv > q): skip the step
                # - same chunk (kv == q): triangular mask inside the chunk
                # - KV chunk in the past (kv < q): every position is visible
                if causal and kv_chunk_idx > q_chunk_idx:
                    continue

                causal_mask = None
                if causal and kv_chunk_idx == q_chunk_idx:
                    n_local = q_chunks[device_id].shape[0]
                    causal_mask = torch.triu(
                        torch.ones(n_local, n_local, dtype=torch.bool, device=Q.device),
                        diagonal=1
                    )

                # Step 0 is always the diagonal chunk, so every query row sees at
                # least its own key first and running_max never stays -inf.
                running_max[device_id], running_sum[device_id], running_output[device_id] = ring_step(
                    local_q=q_chunks[device_id],
                    received_k=k_chunks[device_id],
                    received_v=v_chunks[device_id],
                    running_max=running_max[device_id],
                    running_sum=running_sum[device_id],
                    running_output=running_output[device_id],
                    scale=scale,
                    causal_mask=causal_mask,
                )

            # Device i sends its K,V to device (i+1) % n
            k_chunks = self._rotate_ring(k_chunks)
            v_chunks = self._rotate_ring(v_chunks)
            kv_source = self._rotate_ring(kv_source)

        # Final normalization (once, at the end), then reassemble the sequence
        output_chunks = [running_output[i] / running_sum[i] for i in range(self.num_devices)]
        return torch.cat(output_chunks, dim=0)

print("RingAttention class defined.")

In [ ]:
# Validation
print("=" * 60)
print("VALIDATING RING ATTENTION")
print("=" * 60)

ring_attn = RingAttention(num_devices=num_devices)

# Test 1: Non-causal attention
print("\n--- Test 1: Non-Causal Attention ---")
ref_output = standard_attention(Q, K, V, causal=False)
ring_output = ring_attn.forward(Q, K, V, causal=False)
max_err = (ring_output - ref_output).abs().max().item()
print(f"Max absolute error: {max_err:.2e}")
assert torch.allclose(ring_output, ref_output, atol=1e-5), "Non-causal ring attention FAILED!"
print("PASSED")

# Test 2: Causal attention
print("\n--- Test 2: Causal Attention ---")
ref_causal = standard_attention(Q, K, V, causal=True)
ring_causal = ring_attn.forward(Q, K, V, causal=True)
max_err_causal = (ring_causal - ref_causal).abs().max().item()
print(f"Max absolute error: {max_err_causal:.2e}")
assert torch.allclose(ring_causal, ref_causal, atol=1e-5), "Causal ring attention FAILED!"
print("PASSED")

# Test 3: Different number of devices
print("\n--- Test 3: Different Device Counts ---")
for n_dev in [2, 4, 8]:
    ring = RingAttention(num_devices=n_dev)
    out = ring.forward(Q, K, V, causal=False)
    err = (out - ref_output).abs().max().item()
    assert torch.allclose(out, ref_output, atol=1e-5), f"Failed for {n_dev} devices"
    print(f"  {n_dev} devices: max_err={err:.2e} PASSED")

# Test 4: Different device counts with causal masking
print("\n--- Test 4: Causal with Different Device Counts ---")
for n_dev in [2, 4, 8]:
    ring = RingAttention(num_devices=n_dev)
    out = ring.forward(Q, K, V, causal=True)
    err = (out - ref_causal).abs().max().item()
    assert torch.allclose(out, ref_causal, atol=1e-5), f"Causal failed for {n_dev} devices"
    print(f"  {n_dev} devices (causal): max_err={err:.2e} PASSED")

# Test 5: Sequence length not divisible by the number of devices
print("\n--- Test 5: Uneven Chunks ---")
Q5, K5, V5 = Q[:50], K[:50], V[:50]
for causal in (False, True):
    ref5 = standard_attention(Q5, K5, V5, causal=causal)
    out5 = RingAttention(num_devices=4).forward(Q5, K5, V5, causal=causal)
    assert torch.allclose(out5, ref5, atol=1e-5), f"Uneven chunks failed (causal={causal})"
    print(f"  seq_len=50, 4 devices, causal={causal}: max_err={(out5 - ref5).abs().max().item():.2e} PASSED")

# Test 6: Memory analysis
print("\n--- Test 6: Memory Analysis ---")
print(f"Full KV memory: {seq_len * head_dim * 2} elements (K + V)")
print()
for n_dev in [1, 2, 4, 8]:
    full_kv_mem = seq_len * head_dim * 2  # K + V, full
    per_device_kv = (seq_len // n_dev) * head_dim * 2  # local chunk
    ring_kv = per_device_kv * (2 if n_dev > 1 else 1)  # chunk computed on + chunk being received
    print(f"  {n_dev} devices: per-device KV = {ring_kv} elements "
          f"({ring_kv / full_kv_mem * 100:.0f}% of full, "
          f"{full_kv_mem / ring_kv:.1f}x reduction)")

# Test 7: Verify ring communication pattern
print("\n--- Test 7: Ring Communication Pattern ---")
ring_4 = RingAttention(num_devices=4)
test_chunks = [f"chunk_{i}" for i in range(4)]
print(f"Initial:  {test_chunks}")
for step in range(4):
    test_chunks = ring_4._rotate_ring(test_chunks)
    print(f"Step {step+1}:   {test_chunks}")
# After 4 rotations, should be back to original
assert test_chunks == [f"chunk_{i}" for i in range(4)], "Ring rotation error!"
print("Ring completes full cycle. PASSED")

print("\nAll tests passed!")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("ring-attention", ring_step=ring_step, RingAttention=RingAttention)